# SWE and SME from scratch — symbolic walkthrough

Derivation of the Shallow-Water Equations (SWE, level=0) and the Shallow
Moment Equations (SME, level>0) directly from incompressible Navier-Stokes.

Everything is mutation-based:

* ``model.apply(op)``  — mutates every equation of the system, returns ``self``.
* ``model.<eq>.apply(op).simplify()`` — chainable in-place mutation on a
  single equation through the proxy.
* ``model.<eq>.solve_for(var)``  — returns an Expression that ``apply``
  consumes directly as ``{var: solution}``.
* ``model.<eq>.remove()`` — drops an equation from the system.

No ``model.equations[name] = model.equations[name].apply(...)`` pattern.
No ``DepthIntegrate`` / ``HydrostaticPressure`` / ``ApplyKinematicBCs`` /
``StressFreeSurface`` / ``ZeroAtmosphericPressure`` / ``SimplifyIntegrals``
shortcuts — everything goes through ``Integrate`` and substitution dicts.

## Imports

The path bootstrap below makes sure we import ``zoomy_core`` from the
worktree we're actually sitting in, not from whatever editable install
Python's site-packages happens to point at (different worktrees can be
at different commits).

In [1]:
import sys
from pathlib import Path

# -- Pin zoomy_core to the current worktree -----------------------------------
#
# zoomy_core is installed as a PEP 660 editable package.  Its finder is
# registered on ``sys.meta_path`` with a hardcoded ``MAPPING`` that points
# at whichever worktree ``pip install -e`` was run from.  That finder
# runs BEFORE ``sys.path``, so a bare ``sys.path.insert`` is silently
# ignored.  We locate the enclosing worktree and rewrite the finder's
# MAPPING + clear any cached module so the next ``import zoomy_core``
# picks up THIS worktree's code.

_here = Path.cwd()
while _here != _here.parent and not (_here / "library" / "zoomy_core" / "zoomy_core").exists():
    _here = _here.parent
_pkg_dir   = _here / "library" / "zoomy_core"
_pkg_inner = _pkg_dir / "zoomy_core"
assert _pkg_inner.exists(), f"could not find library/zoomy_core/zoomy_core from {Path.cwd()}"

# Drop any cached modules so the next import is fresh.
for _k in list(sys.modules):
    if _k == "zoomy_core" or _k.startswith("zoomy_core."):
        del sys.modules[_k]

# Patch every editable finder on sys.meta_path whose module defines a
# MAPPING that mentions zoomy_core.  (Each zoomy_* package has its own
# finder; we only touch zoomy_core's.)
_patched = False
for _finder in sys.meta_path:
    _mod_name = getattr(_finder, "__module__", "") or ""
    _mod = sys.modules.get(_mod_name)
    _mapping = getattr(_mod, "MAPPING", None) if _mod is not None else None
    if isinstance(_mapping, dict) and "zoomy_core" in _mapping:
        _mapping["zoomy_core"] = str(_pkg_inner)
        _patched = True

# Belt-and-braces: also prepend to sys.path.
if str(_pkg_dir) not in sys.path:
    sys.path.insert(0, str(_pkg_dir))

import sympy as sp
import zoomy_core
from zoomy_core.model.models.ins_generator import (
    StateSpace, FullINS, Integrate, Newtonian,
)
from zoomy_core.model.models.sme_model import hydrostatic_scaling

print("worktree root:", _here)
print("zoomy_core.__file__:", zoomy_core.__file__)
print("editable MAPPING patched:", _patched)

worktree root: /mnt/userdrive/Users/home/adam-obbpb5az1dhsjzf/git/Zoomy-symbolic
zoomy_core.__file__: /mnt/userdrive/Users/home/adam-obbpb5az1dhsjzf/git/Zoomy-symbolic/library/zoomy_core/zoomy_core/__init__.py
editable MAPPING patched: True


## Step 1 — Start from the raw Navier-Stokes system

In [2]:
state = StateSpace(dimension=2)          # (t, x, z)
model = FullINS(state)
model.describe()

**INS** (continuity, momentum.x, momentum.z)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**momentum.x:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial z} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial x} u^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial x} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{xx}{\left(t,x,z \right)}}{\rho} - \frac{\frac{\partial}{\partial z} \tau_{xz}{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$

**momentum.z:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} w{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial x} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial z} w^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial z} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{zx}{\left(t,x,z \right)}}{\rho} - \frac{\frac{\partial}{\partial z} \tau_{zz}{\left(t,x,z \right)}}{\rho}}_{stress} \\
  & + \underbrace{g}_{source}
  &= 0
\end{aligned}
$$


## Step 2 — Hydrostatic assumption on z-momentum

$w = 0$, $\tau_{zz} = \tau_{xz} = \tau_{zx} = 0$ inside z-momentum only.
Chained: ``apply(...)``→proxy, ``.simplify()``→proxy.

In [3]:
model.momentum.z.apply(hydrostatic_scaling(state)).simplify()
model.momentum.z.describe()

**z_momentum** (7 terms)

$$
\begin{aligned}
  & \underbrace{\frac{d}{d t} 0}_{temporal} \\
  & + \underbrace{\frac{d}{d x} 0 + \frac{d}{d z} 0}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial z} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{d}{d x} 0}{\rho} - \frac{\frac{d}{d z} 0}{\rho}}_{stress} \\
  & + \underbrace{g}_{source}
  &= 0
\end{aligned}
$$

## Step 3 — Integrate z-momentum analytically to get $p(z)$

Integrate $g + \partial_z p / \rho = 0$ from the current depth $z$ up to the
free surface $\eta$.  `method="analytical"` runs ``sympy.integrate`` on the
whole expression (needed for partial / running integrals).

In [4]:
model.momentum.z.apply(
    Integrate(state.z, state.z, state.eta, method="analytical")
)
model.momentum.z.describe()

**z_momentum** (20 terms)

$$
\begin{aligned}
  & \underbrace{- z \frac{d}{d t} 0 + b{\left(t,x \right)} \frac{d}{d t} 0 + h{\left(t,x \right)} \frac{d}{d t} 0}_{temporal} \\
  & \underbrace{- z \frac{d}{d x} 0 - z \frac{d}{d z} 0 + b{\left(t,x \right)} \frac{d}{d x} 0 + b{\left(t,x \right)} \left. \frac{d}{d Dummy_{74}} 0 \right|_{\substack{ Dummy_{74}=b{\left(t,x \right)} + h{\left(t,x \right)} }} + h{\left(t,x \right)} \frac{d}{d x} 0 + h{\left(t,x \right)} \left. \frac{d}{d Dummy_{74}} 0 \right|_{\substack{ Dummy_{74}=b{\left(t,x \right)} + h{\left(t,x \right)} }}}_{convection} \\
  & \underbrace{- \frac{p{\left(t,x,z \right)}}{\rho} + \frac{p{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)}}{\rho}}_{pressure} \\
  & + \underbrace{\frac{z \frac{d}{d x} 0}{\rho} + \frac{z \frac{d}{d z} 0}{\rho} - \frac{b{\left(t,x \right)} \frac{d}{d x} 0}{\rho} - \frac{b{\left(t,x \right)} \left. \frac{d}{d Dummy_{74}} 0 \right|_{\substack{ Dummy_{74}=b{\left(t,x \right)} + h{\left(t,x \right)} }}}{\rho} - \frac{h{\left(t,x \right)} \frac{d}{d x} 0}{\rho} - \frac{h{\left(t,x \right)} \left. \frac{d}{d Dummy_{74}} 0 \right|_{\substack{ Dummy_{74}=b{\left(t,x \right)} + h{\left(t,x \right)} }}}{\rho}}_{stress} \\
  & \underbrace{- g z + g b{\left(t,x \right)} + g h{\left(t,x \right)}}_{source}
  &= 0
\end{aligned}
$$

## Step 4 — Atmospheric-pressure BC at the free surface

$p(\eta) = 0$ (atmospheric gauge).  Plain substitution dict.

In [5]:
model.momentum.z.apply({state.p.subs(state.z, state.eta): 0}).simplify()
model.momentum.z.describe()

**z_momentum** (15 terms)

$$
\begin{aligned}
  & \underbrace{- z \frac{d}{d t} 0 + b{\left(t,x \right)} \frac{d}{d t} 0 + h{\left(t,x \right)} \frac{d}{d t} 0}_{temporal} \\
  & \underbrace{- z \frac{d}{d x} 0 - z \frac{d}{d z} 0 + b{\left(t,x \right)} \frac{d}{d x} 0 + h{\left(t,x \right)} \frac{d}{d x} 0}_{convection} \\
  & \underbrace{- \frac{p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & + \underbrace{\frac{z \frac{d}{d x} 0}{\rho} + \frac{z \frac{d}{d z} 0}{\rho} - \frac{b{\left(t,x \right)} \frac{d}{d x} 0}{\rho} - \frac{h{\left(t,x \right)} \frac{d}{d x} 0}{\rho}}_{stress} \\
  & \underbrace{- g z + g b{\left(t,x \right)} + g h{\left(t,x \right)}}_{source}
  &= 0
\end{aligned}
$$

## Step 5 — Substitute the solved $p$ into x-momentum, then drop z-momentum

In [6]:
model.momentum.x.apply(model.momentum.z.solve_for(state.p)).simplify()
model.momentum.z.remove()
model.describe()

**INS** (continuity, momentum.x)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**momentum.x:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial z} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial x} u^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{g \frac{\partial}{\partial x} b{\left(t,x \right)} + g \frac{\partial}{\partial x} h{\left(t,x \right)}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{xx}{\left(t,x,z \right)}}{\rho} - \frac{\frac{\partial}{\partial z} \tau_{xz}{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$


## Step 6 — Newtonian constitutive model

System-level mutation: ``model.apply(op)`` runs the op on every equation
of the system, returns ``self``, so ``.simplify()`` chains.
The internal simplify is linearity-only (``d(b+h)/dx → db/dx + dh/dx``)
and intentionally does **not** chain-rule-expand conservative forms like
``∂_x(u²)``, so they survive for the Leibniz integration in Step 7.

In [7]:
model.apply(Newtonian(state)).simplify()
model.describe()

**INS** (continuity, momentum.x)

**Assumptions:** Newtonian

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**momentum.x:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial z} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial x} u^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{g \frac{\partial}{\partial x} b{\left(t,x \right)} + g \frac{\partial}{\partial x} h{\left(t,x \right)}}_{pressure} \\
  & \underbrace{- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)} - \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)} - \frac{\frac{\partial}{\partial x} 2 \nu \rho \frac{\partial}{\partial x} u{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$


## Step 7 — Depth-integrate continuity and x-momentum from $b$ to $\eta$

One ``Integrate`` call at system level — per-term auto dispatch picks
Leibniz for $\partial_x$ and the fundamental theorem for $\partial_z$.

In [8]:
model.apply(Integrate(state.z, state.b, state.eta, method="auto"))
model.continuity.describe()

**continuity** (5 terms)

$$
- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz + \left. w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \left. w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} = 0
$$

In [9]:
model.momentum.x.describe()

**x_momentum** (19 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{\partial}{\partial t} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial t} b{\left(t,x \right)} \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u^{2}{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. u^{2}{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u^{2}{\left(t,x,z \right)}\, dz + \left. u{\left(t,x,z \right)} w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \left. u{\left(t,x,z \right)} w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }}}_{convection} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. g b{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. g h{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. g b{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. g h{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. - 2 \nu \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. - 2 \nu \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial}{\partial x} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 8 — Resolve $w$ boundary terms via the kinematic BCs

Two substitutions for the $w$ evaluations at bottom and surface.  System-level
``apply(dict)`` applies the dict to every equation and chains into ``simplify``.

In [10]:
u_at_b = state.u.subs(state.z, state.b)
u_at_eta = state.u.subs(state.z, state.eta)
kinematic_bcs = {
    state.w.subs(state.z, state.b):
        sp.Derivative(state.b, state.t) + u_at_b * sp.Derivative(state.b, state.x),
    state.w.subs(state.z, state.eta):
        sp.Derivative(state.eta, state.t) + u_at_eta * sp.Derivative(state.eta, state.x),
}
model.apply(kinematic_bcs).simplify()
model.momentum.x.describe()

**x_momentum** (12 terms)

$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{2 \nu \frac{\partial}{\partial x} b{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - 2 \nu \frac{\partial}{\partial x} b{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + 2 \nu \frac{\partial}{\partial x} h{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial}{\partial x} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 9 — Zero tangential stress at surface and bottom

Stress-free surface ($\tau_{xz}|_\eta = 0$) and zero tangential normal
stress at both boundaries ($\tau_{xx}|_b = \tau_{xx}|_\eta = 0$).

In [11]:
stress_free_surface = {state.tau["xz"].subs(state.z, state.eta): 0}
no_tangential_normal_stress = {
    state.tau["xx"].subs(state.z, state.b): 0,
    state.tau["xx"].subs(state.z, state.eta): 0,
}
model.apply(stress_free_surface).apply(no_tangential_normal_stress).simplify()
model.momentum.x.describe()

**x_momentum** (12 terms)

$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{2 \nu \frac{\partial}{\partial x} b{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - 2 \nu \frac{\partial}{\partial x} b{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + 2 \nu \frac{\partial}{\partial x} h{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial}{\partial x} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 10 — Bottom stress closure (Navier slip)

$\tau_{xz}|_b = \rho\,(\lambda/\tau_c)\,u|_b$.  Dict + chain.

In [12]:
lamda = sp.Symbol("lamda", positive=True)
tau_c = sp.Symbol("tau_c", positive=True)
friction_closure = {
    state.tau["xz"].subs(state.z, state.b): state.rho * (lamda / tau_c) * u_at_b,
}
model.apply(friction_closure).simplify()
model.momentum.x.describe()

**x_momentum** (12 terms)

$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{2 \nu \frac{\partial}{\partial x} b{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - 2 \nu \frac{\partial}{\partial x} b{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + 2 \nu \frac{\partial}{\partial x} h{\left(t,x \right)} \left. \frac{\partial}{\partial x} u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial}{\partial x} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## What's left on the board

At this point every equation is:

* Depth-integrated over $[b,\eta]$.
* Closed for $w$, the tangential normal stress, and bottom shear.
* In terms of the velocity field $u(t,x,z)$, its surface/bottom evaluations
  $u|_b$, $u|_\eta$, and volume integrals $\int_b^\eta f\,dz$.

The remaining step is projection against a vertical basis:

* **SWE (level=0)** — constant vertical profile.  Substitute
  ``u(t,x,z) → u_mean(t,x)`` (both in the volume integrals and in the
  surface/bottom evaluations) and evaluate the resulting integrals.
* **SME (level≥1)** — expand ``u(t,x,z) = Σ α_k(t,x) φ_k(ζ)`` and Galerkin-
  test against each ``φ_l``.  Today this is ``Expression.project_onto_basis``;
  it rewrites ``Integral`` nodes and we complement it with explicit
  ``{u|_b: Σ α_k φ_k(0), u|_η: Σ α_k φ_k(1)}`` substitutions for the
  boundary evaluations.

Both projections follow the same mutation pattern used above:
``model.apply(...)`` + chained ``.simplify()``.